# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# To examine full metadata (not as dict!), we convert to dict for printing
metadata_dict = dataset.metadata.to_json()

print(f"Dataset: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")
print(f"Published: {metadata_dict['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema provides data structures called _record sets_ (`RecordSet`), representing logical tables, along with their fields and column mappings. We'll print available RecordSet and Field `@id`s.

In [ ]:
# List all available record sets and their associated fields
print("Available RecordSets, Fields, and Columns:")
for record_set in dataset.record_sets:
    print(f"\nRecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'No name')}")
    print(f"  Description: {record_set.get('description', '')}")
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            # Get field definition
            if isinstance(field, dict):
                field_id = field.get('@id', str(field))
            else:
                field_id = str(field)
            try:
                field_obj = dataset.field(field_id)
                print(f"    - Field @id: {field_id}")
                print(f"      name: {field_obj.get('name', 'No name')}")
                if 'column' in field_obj:
                    print(f"      column(s): {[c['@id'] if isinstance(c,dict) and '@id' in c else c for c in field_obj['column']]}")
            except Exception as e:
                print(f"    - Field @id: {field_id} (definition not found)")
    else:
        print("  No field listed.")

To see sample records, pick a RecordSet's `@id` identified above.

For example, if the `@id` is `/tables/0`, use it as below. Here we enumerate records for the first RecordSet.

In [ ]:
# Replace with the desired RecordSet @id identified above or explore all
print("\nSample record preview from available RecordSets:")
for record_set in dataset.record_sets:
    record_set_id = record_set['@id']
    print(f"\n--- Records for RecordSet @id: {record_set_id} ---")
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            pprint.pprint(record)
            if i >= 2: break
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

_Note: All field, column, and record set references must use their `@id`._

In [ ]:
# Collect all RecordSet @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Extract data from each RecordSet as a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Pick a main data RecordSet (assume first one with data)
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set_id = rid
        break
if main_record_set_id is None:
    raise ValueError('No record sets with tabular data found.')

print(f"Fields of RecordSet @id {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes sample code operating on fields and values _by their `@id` columns only_.

In [ ]:
df = dataframes[main_record_set_id]
print(f"\nAvailable fields in RecordSet {main_record_set_id}:")
print(list(df.columns))

# Try to pick a numeric field. We'll pick the first float-compatible column.
# In real EDA, replace with the correct @id for your analysis (e.g., '/tables/0/fields/abundance')
numeric_field_id = None
for col in df.columns:
    # Try to infer if this column is numeric
    try:
        df[col] = pd.to_numeric(df[col], errors='ignore')
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    print("No numeric fields found.")
else:
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    # Example filtering: filter for values greater than mean
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
        / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (pick first non-numeric)
    group_field_id = None
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution and a grouped bar plot if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        mean_per_group = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:10]
        sns.barplot(y=mean_per_group.index.astype(str), x=mean_per_group.values, orient='h')
        plt.xlabel(f"Mean of {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 10)")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells using the `mlcroissant` library. We explored the available record sets and fields by their `@id`, extracted records into DataFrames, filtered and normalized a numeric field by its Croissant `@id`, and visualized its distribution. For further analysis, refer back to the field and column `@id` mappings discovered for more domain-specific exploration.